# Measure selection and read latency

Compare Paper greedy with tiles at coarse widths and successive halvings down to one-row Top-R.
Each method uses the same captured importance, requested row budget and direct-I/O reader.

Inputs are the profile from 01, traces from 02 and settings from
[03_01](03_01_tune_paper.ipynb). Run from this directory on the target SSD with Linux,
CUDA, PyTorch, a C++ compiler and the upstream dependencies.

The measurement begins with prepared importance on the CPU and ends with selected bytes
on the CPU. It uses a random file sized for projection weights. GPU weight transfer,
model computation and answer accuracy are outside this experiment.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter_ns
import json
import os
import platform
import subprocess
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.cpp_extension import load

UPSTREAM = Path("../../vlm-flash")
sys.path.insert(0, str((UPSTREAM / "src").resolve()))
sys.path.insert(0, str((UPSTREAM / "scripts").resolve()))
import vlmflash
from vlmflash._native import native
from _lock import exclusive

## 1. Inputs and settings

Take up to 32 evenly spaced traces per shape, preserving archive order within each shape.
For fraction $f$, select at most $R=\max(1,\lfloor fN\rfloor)$ rows.
Tiles fill the budget exactly; Paper may underfill it. At $R=N$, all methods bypass selection.

In [ ]:
PROFILE = Path("../01/chunk_latency_measurements.csv")
TRACE = Path("../02/activation_traces.json")
TUNING = Path("paper_tuning.csv")
MAX_TRACES_PER_SHAPE = 32  # 0 uses all archived traces.
FRACTIONS = np.arange(3, 11) / 10
WARMUP, REPEATS, THREADS = 3, 30, 6
MAX_READ_KIB = 768
COARSE_SCALES = [1, 1.25, 1.5, 1.75, 2]
PAPER = "Paper greedy (GPU sort)"

In [ ]:
archive = json.loads(TRACE.read_text())
weight_bytes = archive["weight_bytes"]
shapes = sorted({(t["n"], t["d"]) for t in archive["traces"]})
traces = []
for shape in shapes:
    group = [t for t in archive["traces"] if (t["n"], t["d"]) == shape]
    count = min(len(group), MAX_TRACES_PER_SHAPE or len(group))
    traces.extend(group[i] for i in np.linspace(0, len(group) - 1, count, dtype=int))

tuning = pd.read_csv(TUNING).set_index(["n", "d"], verify_integrity=True)
assert set(shapes) <= set(tuning.index), "Run 03_01 for the missing projection shapes."
curve = pd.read_csv(PROFILE).groupby("size_kib").latency_ms.median().sort_index()
assert np.isfinite(curve).all() and (curve > 0).all()
boundary = float((curve / curve.index).idxmin())
ceiling = float(curve.index.max())
table = vlmflash.LatencyTable(curve.to_dict())

## 2. Native implementations

Paper ranks multiscale windows by importance per estimated cost on the GPU, then greedily
accepts nonoverlapping windows on the CPU. Its GPU sorting and score transfers are timed.
Tiles rank full tiles on the CPU, fill a residual interval and compare offsets zero and half a tile.
[selectors.cpp](selectors.cpp) contains the original tile implementation unchanged.
Build both implementations before measuring.

In [ ]:
os.environ.setdefault("MAX_JOBS", "1")
torch.set_num_threads(1)
assert torch.cuda.is_available(), "Paper's measured GPU sort requires CUDA."
reader = native()
assert reader is not None, "Build the upstream native extension first."
selectors = load("vlmflash_selection03_notebook", [str(Path("selectors.cpp").resolve())],
                 extra_cflags=["-O3", "-std=c++17"], verbose=True)

## 3. Candidate geometry

Let $s$ minimize measured cost per KiB. Convert it to rows before taking coarse multiples
$1,1.25,1.5,1.75,2$ and successive halvings down to one row. Keep each integer width once.
This profile boundary differs from the fitted two-line boundary used in 02.

The cost lookup interpolates the measured profile and sums costs over maximal selected runs.
It chooses masks; direct-I/O times are measured separately with six reader threads.

In [ ]:
def tile_grid(s):
    levels = max(0, int(np.ceil(np.log2(max(1, s)))))
    scales = [(f"{a:g}s", float(a)) for a in COARSE_SCALES]
    scales += [(f"s/2^{j}", 2.**-j) for j in range(1, levels + 1)]
    seen, grid = set(), []
    for label, scale in scales:
        width = max(1, int(np.ceil(scale * s)))
        if width not in seen:
            name = "Top-R" if width == 1 else f"Tiles ({label})"
            grid.append((name, scale, width))
            seen.add(width)
    return grid

In [ ]:
def paper_windows(n, w, tuned):
    start = max(1, int(tuned.start_kb / w))
    stride = max(1, int(tuned.jump_cap_kb / w))
    if ceiling < 2*w:
        return torch.tensor([1]), torch.tensor([table.read_ms(w)], dtype=torch.float64), 1
    sizes = list(range(start, min(n+1, int((ceiling+1)/w)), start))
    assert sizes, "Paper window range is empty."
    return (torch.tensor(sizes),
            torch.tensor([table.read_ms(w*r) for r in sizes], dtype=torch.float64), stride)

In [ ]:
def prepare(trace):
    values = torch.tensor(trace["values"], dtype=torch.float32)
    assert values.ndim == 1 and torch.isfinite(values).all() and (values >= 0).all()
    n, d, row_bytes = trace["n"], trace["d"], trace["row_bytes"]
    row_kib = row_bytes / 1024
    costs = torch.tensor([0.] + [table.read_ms(row_kib * r) for r in range(1, n + 1)],
                         dtype=torch.float64)
    sizes, delays, stride = paper_windows(n, row_kib, tuning.loc[(n, d)])
    methods = {PAPER: lambda R: reader.select_chunks(values, R, sizes, delays, stride, True)[0]}
    grid = tile_grid(boundary / row_kib)
    for name, scale, width in grid:
        methods[name] = lambda R, B=width: selectors.tiles(values, R, costs, B)[0]
    return values, costs, methods, grid

## 4. Case information

A case is one trace, requested budget and method. Save its mask and metadata once.
Retained importance is $100\sum_i v_iM_i/\sum_i v_i$; fill is $100|M|/R$.
Importance is the captured mean absolute activation, a proxy for answer quality.
Intervals are maximal contiguous runs stored as `[start, end)` pairs.

In [ ]:
def describe(mask, values, R, costs, method):
    selected, total = int(mask.sum()), float(values.double().sum())
    assert selected <= R if method == PAPER else selected == R
    spans = np.flatnonzero(np.diff(np.r_[False, mask.numpy().astype(bool), False])).reshape(-1, 2)
    return dict(R_selected=selected, fill_pct=100 * selected / R,
                retained_pct=100 * float(values[mask.bool()].double().sum()) / total if total else np.nan,
                modeled_io_ms=float(costs[np.diff(spans).ravel()].sum()), runs=len(spans),
                intervals=json.dumps(spans.tolist()))

## 5. Timing

Measure selection and selection-through-read wall time with the same outer clock.
Native `io_ms` covers its read/thread interval; wall time also includes read preparation.
Adjacent rows merge, and requests split at 768 KiB. Require `O_DIRECT` so the measurements
bypass the operating system's page cache. Validation occurs after the timed interval.

In [ ]:
def timed_read(select, R, expected, blob, row_bytes):
    full = R == expected.numel()
    begin = perf_counter_ns()
    mask = expected if full else select(R)
    after_select = perf_counter_ns()
    raw, io_us, _, direct = reader.read_rows(
        blob, mask, row_bytes, "cpu", THREADS, MAX_READ_KIB * 1024, True)
    end = perf_counter_ns()
    assert direct, "O_DIRECT unavailable; refusing page-cache timings."
    assert torch.equal(mask, expected) and raw.shape == (int(expected.sum()), row_bytes)
    return dict(selection_ms=(after_select - begin) / 1e6,
                io_ms=io_us / 1000, wall_ms=(end - begin) / 1e6)

In [ ]:
def measure_trace(trace, blob, rng):
    values, costs, methods, grid = prepare(trace)
    n = len(values)
    dense = torch.ones(n, dtype=torch.uint8)
    widths = {name: (scale, width) for name, scale, width in grid}

    for fraction in FRACTIONS:
        R = max(1, int(round(float(fraction), 6) * n))
        checked = {}
        for name, select in methods.items():
            mask = dense if R == n else select(R)
            scale, width = widths.get(name, (np.nan, np.nan))
            if width == 1:
                top = torch.zeros(n, dtype=torch.uint8)
                top[torch.argsort(values, descending=True, stable=True)[:R]] = 1
                assert torch.equal(mask, top), "One-row tiles must equal stable Top-R."
            case_id = len(case_records)
            case_records.append(dict(case_id=case_id, trace_id=trace["trace_id"], n=n, d=trace["d"],
                fraction=float(fraction), R_nominal=R, tile_scale=scale, tile_rows=width,
                tile_kib=width * trace["row_bytes"] / 1024, method=name,
                **describe(mask, values, R, costs, name)))
            checked[name] = (case_id, mask.clone())

        for repeat in range(-WARMUP, REPEATS):
            for name in rng.permutation(list(methods)):
                case_id, mask = checked[name]
                timing = timed_read(methods[name], R, mask, blob, trace["row_bytes"])
                if repeat >= 0:
                    timing_records.append(dict(case_id=case_id, repeat=repeat, **timing))

## 6. Measurement

Allocate and flush a random file on the target filesystem before timing.
Randomize method order within each trace and budget, including warmup rounds.
The lock prevents cooperating experiments from measuring this device concurrently.

In [ ]:
def write_blob(path, size):
    with open(path, "wb") as stream:
        os.posix_fallocate(stream.fileno(), 0, size)
        for begin in range(0, size, 2**20):
            stream.write(os.urandom(min(2**20, size - begin)))
        stream.flush()
        os.fsync(stream.fileno())

In [ ]:
rng = np.random.default_rng(0)
case_records, timing_records = [], []
with exclusive("experiment03"), TemporaryDirectory(dir=Path.cwd()) as work:
    blob = str(Path(work) / "rows.dat")
    write_blob(blob, max(t["n"] * t["row_bytes"] for t in traces))
    for index, trace in enumerate(traces, 1):
        measure_trace(trace, blob, rng)
        if index % 16 == 0 or index == len(traces):
            print(f"{index}/{len(traces)} traces measured", flush=True)

## 7. Output

`cases.csv` contains one row per case; `timings.csv` contains its repeat samples.
`case_id` joins the two files. Store acquisition settings and the selected tuning snapshot
in `environment.json`, so later tuning changes do not change interpretation of this run.
The save cells replace these three files only after the measurement completes.

In [ ]:
cases = pd.DataFrame(case_records)
timings = pd.DataFrame(timing_records)
assert cases.case_id.is_unique
assert not timings.duplicated(["case_id", "repeat"]).any()
assert timings.groupby("case_id").size().eq(REPEATS).all()

paper_tuning = []
for n, d in shapes:
    setting = tuning.loc[(n, d)]
    paper_tuning.append(dict(n=n, d=d,
        best=dict(start_kb=int(setting.start_kb), jump_cap_kb=int(setting.jump_cap_kb),
                  overhead_ms=float(setting.overhead_ms)),
        num_feasible=int(setting.num_feasible), num_evaluated=int(setting.num_evaluated)))

In [ ]:
environment = dict(
    platform=platform.platform(), torch=torch.__version__, cuda_available=torch.cuda.is_available(),
    paper_sort_devices=["cuda"], paper_tuning=paper_tuning,
    torch_threads=torch.get_num_threads(), reader_threads=THREADS, direct_io=True, destination="cpu",
    tile_boundary_kib=boundary, coarse_scales=COARSE_SCALES, dtype_bytes=weight_bytes,
    tile_grid="halving_to_one_row; unique widths per shape", tile_offsets="0, floor(B/2)",
    paper_ceiling_kib=ceiling, traces=len(traces), warmup=WARMUP, repeats=REPEATS,
    fractions=FRACTIONS.tolist(), max_read_kib=MAX_READ_KIB,
    measurement="prepared CPU importance through selected bytes on CPU; no model compute",
    upstream_commit=subprocess.check_output(["git", "-C", str(UPSTREAM), "rev-parse", "HEAD"],
                                            text=True).strip(),
    mount=subprocess.check_output(["findmnt", "-T", str(Path.cwd()), "-n", "-o", "SOURCE,FSTYPE,TARGET"],
                                  text=True).strip(),
    cases_file="cases.csv", timings_file="timings.csv", cases=len(cases), samples=len(timings))

In [ ]:
cases.to_csv("cases.csv", index=False)
timings.to_csv("timings.csv", index=False)
Path("environment.json").write_text(json.dumps(environment, indent=2) + "\n")
print(f"Saved {len(cases):,} cases and {len(timings):,} repeat samples.")